# Pêndulo Simples

Vamos resolver a equação do pêndulo com 4 métodos. Em cada bloco, o código calcula a solução, mostra alguns valores e plota `theta(t)`.


Equação do problema:

`theta'' + (g/L) sin(theta) = 0`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import odeint
try:
    from IPython.display import HTML
except Exception:
    HTML = None

g, L = 9.81, 1.0                               # parâmetros do sistema
theta0, omega0 = np.deg2rad(20.0), 0.0         # condição inicial
dt = 0.01                                      # passo de tempo
t = np.arange(0.0, 10.0 + dt, dt)             # pontos de tempo
N = len(t)                                     # número de pontos

def mostrar(nome, y):
    print(f"\nSaída numérica - {nome}")
    print(" i   t(s)    theta(rad)")
    for i in range(10):
        print(f"{i:2d}  {t[i]:6.3f}   {y[i]:10.6f}")
    plt.figure(figsize=(8, 4))
    plt.plot(t, y, lw=2)
    plt.xlabel('tempo (s)')
    plt.ylabel('theta(t)')
    plt.title(nome)
    plt.grid(alpha=0.3)
    plt.show()


## Método 1: Euler


In [ ]:
def euler():
    th = np.zeros(N)                           # ângulo
    om = np.zeros(N)                           # velocidade angular
    th[0], om[0] = theta0, omega0              # condição inicial
    for i in range(N - 1):
        a = -(g / L) * np.sin(th[i])           # aceleração angular
        th[i + 1] = th[i] + om[i] * dt         # novo theta
        om[i + 1] = om[i] + a * dt             # novo omega
    return th, om

th_eu, om_eu = euler()
mostrar('Euler', th_eu)


## Método 2: Euler-Cromer


In [ ]:
def euler_cromer():
    th = np.zeros(N)                           # ângulo
    om = np.zeros(N)                           # velocidade angular
    th[0], om[0] = theta0, omega0              # condição inicial
    for i in range(N - 1):
        a = -(g / L) * np.sin(th[i])           # aceleração angular
        om[i + 1] = om[i] + a * dt             # novo omega
        th[i + 1] = th[i] + om[i + 1] * dt     # novo theta
    return th, om

th_ec, om_ec = euler_cromer()
mostrar('Euler-Cromer', th_ec)


## Método 3: RK4


In [ ]:
def derivadas(th, om):
    return om, -(g / L) * np.sin(th)           # theta' e omega'

def rk4():
    th = np.zeros(N)                           # ângulo
    om = np.zeros(N)                           # velocidade angular
    th[0], om[0] = theta0, omega0              # condição inicial
    for i in range(N - 1):
        k1t, k1o = derivadas(th[i], om[i])
        k2t, k2o = derivadas(th[i] + 0.5*dt*k1t, om[i] + 0.5*dt*k1o)
        k3t, k3o = derivadas(th[i] + 0.5*dt*k2t, om[i] + 0.5*dt*k2o)
        k4t, k4o = derivadas(th[i] + dt*k3t, om[i] + dt*k3o)
        th[i + 1] = th[i] + (dt/6)*(k1t + 2*k2t + 2*k3t + k4t)
        om[i + 1] = om[i] + (dt/6)*(k1o + 2*k2o + 2*k3o + k4o)
    return th, om

th_rk, om_rk = rk4()
mostrar('RK4', th_rk)


## Método 4: odeint


In [ ]:
def sistema(y, tempo):
    th, om = y                                 # separa o estado
    return [om, -(g / L) * np.sin(th)]         # devolve as derivadas

sol = odeint(sistema, [theta0, omega0], t)    # integração pronta
th_od, om_od = sol[:, 0], sol[:, 1]           # separa theta e omega
mostrar('odeint', th_od)


## Bônus visual

A animação fica separada da parte principal do notebook.


In [ ]:
def indices_animacao(total, max_frames=160):
    return np.unique(np.linspace(0, total - 1, min(max_frames, total), dtype=int))

def animar(theta):
    x = L*np.sin(theta)                        # posição horizontal
    y = -L*np.cos(theta)                       # posição vertical
    fr = indices_animacao(N)                   # menos frames
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
    ax1.set(xlim=(-1.22, 1.22), ylim=(-1.22, 0.22), xlabel='x (m)', ylabel='y (m)', title='Pêndulo')
    ax1.set_aspect('equal', adjustable='box')
    ax1.grid(alpha=0.3)
    ax1.scatter([0.0], [0.0], color='black', s=28)
    ax2.set(xlim=(0, t[-1]), ylim=(1.1*theta.min(), 1.1*theta.max()), xlabel='tempo (s)', ylabel='theta(t)', title='odeint')
    ax2.grid(alpha=0.3)
    haste, = ax1.plot([], [], color='tab:green', lw=2.5)
    massa, = ax1.plot([], [], 'o', color='tab:green', ms=12)
    curva, = ax2.plot([], [], color='tab:green', lw=2)
    def update(i):
        haste.set_data([0.0, x[i]], [0.0, y[i]])
        massa.set_data([x[i]], [y[i]])
        curva.set_data(t[:i + 1], theta[:i + 1])
        return haste, massa, curva
    ani = FuncAnimation(fig, update, frames=fr, interval=20, blit=False)
    fig._ani = ani
    plt.tight_layout()
    return ani

ani = animar(th_od)
html = HTML(ani.to_jshtml())
plt.close(ani._fig)
html
